# ✈️ Airline Satisfaction Prediction — S6E10

 📋 Version 2 — EDA Complete

**Goal:** Determine the best model for this month's competition.

**EDA Findings:**
- **Dataset:** 699,635 train rows / 299,844 test rows / 129,880 original rows
- **Target:** `satisfaction` (False 55.6% / True 44.4%) — balanced, ROC AUC metric
- **Features:** 17 numerical + 5 categorical (low cardinality)
- **Missing:** Only `Arrival Delay in Minutes` (0.04% train/test, 0.3% orig)
- **Numerical:** Age, Flight Distance, 14 service ratings (0-5 scale), 2 delay features
- **Categorical:** Gender (2), Customer Type (2), Type of Travel (2), Class (3)

**Model Choice for This Episode:**
- **LGBM** selected for diversity (XGBoost was used in previous episode)

 🔬 Experiment: LightGBM Baseline

First baseline for S6E10 with LightGBM using raw features + native categorical handling.

**Setup:**
- 5-fold StratifiedKFold
- Dynamic seeds: 123 + fold (124-128)
- Native categorical handling for 4 categorical features
- Raw features only (no feature engineering yet)
- Minimal parameters: lr=0.1, max_depth=6, 1000 trees

| Metric | Value |
|--------|-------|
| OOF AUC | **0.95833** |
| LB | **0.95814** |
| Std Fold AUC | 0.00057 |

**Top Features:** Flight Distance, Age, Inflight wifi service, Checkin service, Online boarding.

**Conclusion:** Strong baseline. LB matches OOF (difference -0.00019) — model is honest, not overfitting.

---

 🔬 Experiment: Digit Features

Added digit features for numerical columns with multiple decimal places (`Age`, `Flight Distance`, `Departure Delay in Minutes`, `Arrival Delay in Minutes`).

**Setup:**
- 5-fold StratifiedKFold
- Dynamic seeds: 123 + fold (124-128)
- LGBM with native categorical handling
- n_estimators=2000, early_stopping=100
- Digit features: 32 new columns (4 columns × 8 positions)

| Metric | Before | After | Gain |
|--------|--------|-------|------|
| OOF AUC | 0.95833 | **0.95902** | **+0.00069** |
| LB | 0.95814 | **0.95874** | **+0.00060** |
| Std Fold AUC | 0.00057 | 0.00056 | — |

**Top Features:** Flight Distance, Age, Inflight wifi service, `Flight Distance_digit0`, `Flight Distance_digit1`.

**Conclusion:** Digit features captured synthetic generation artifacts. Early stopping triggered before 2000 trees — model converged faster than expected.

 🔬 Experiment: Frequency Encoding + Tuned Parameters

Added frequency encoding for all columns and tuned hyperparameters.

**Setup:**
- 5-fold StratifiedKFold
- Dynamic seeds: 123 + fold (124-128)
- LGBM with native categorical handling
- n_estimators=2000, early_stopping=100
- **learning_rate=0.05** (was 0.1)
- **max_depth=7** (was 6)
- Frequency encoding: 21 new columns

| Metric | Before | After | Gain |
|--------|--------|-------|------|
| OOF AUC | 0.95902 | **0.95940** | **+0.00038** |
| LB | 0.95874 | **0.95898** | **+0.00024** |
| Std Fold AUC | 0.00056 | 0.00060 | — |

**Top Features:** Flight Distance, `Flight Distance_freq`, Age, `Age_freq`, Online boarding.

**Conclusion:** Frequency encoding and tuned parameters (lower LR, deeper trees) improved both OOF and LB. Frequency features rank in top-5, showing that value rarity carries real signal.

 🎯 Target Encoding — Key Improvement

Target Encoding was added **inside each fold** (leakage-safe) for 23 columns (21 original + 2 binned features) with three smoothing strengths: `auto`, `10`, and `100`.

| Metric | Before TE | After TE | Gain |
|--------|-----------|----------|------|
| OOF AUC | 0.95940 | **0.95978** | **+0.00038** |
| LB | 0.95898 | **0.95948** | **+0.00050** |

Target Encoding replaces each value with the smoothed mean of the target for that value. Three smoothing levels give the model different views of each feature. Applied inside cross-validation to prevent leakage.

 🔬 Experiment: Tuned Parameters + Smooth Keys

Added regularization and sampling to reduce overfitting, plus 4 new Smooth Keys added to the TE column set.

**Setup:**
- 5-fold StratifiedKFold
- Dynamic seeds: 123 + fold (124-128)
- LGBM with native categorical handling
- n_estimators=2000, early_stopping=100
- **learning_rate=0.03** (was 0.05)
- **max_depth=8** (was 7)
- **Regularization:** lambda_l2=5.0, lambda_l1=0.05, min_child_samples=5
- **Sampling:** bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9
- **Smooth Keys:** 4 new bins for TE (distance_100, distance_500, age_5, dep_delay_10)

| Metric | Before | After | Gain |
|--------|--------|-------|------|
| OOF AUC | 0.95978 | **0.96000** | **+0.00022** |
| LB | Pending | **Pending** | — |
| Std Fold AUC | 0.00061 | 0.00058 | — |

**Top Features:** `Flight Distance_freq`, Flight Distance, Age, `TE_Flight Distance_100`, `TE_Flight Distance_auto`.

**Conclusion:** Regularization and sampling improved stability. Fold 2 still shows a consistent dip (also present in other public baselines), suggesting a data-split artifact rather than seed behavior.

 🔬 Experiment: Numeric-to-Categorical

Converted 17 numerical features into categorical dtype so LGBM handles them as native categories.

**Setup:**
- 5-fold StratifiedKFold
- Dynamic seeds: 123 + fold (124-128)
- LGBM with native categorical handling
- n_estimators=2000, early_stopping=100
- learning_rate=0.03, max_depth=8
- Regularization: lambda_l2=5.0, lambda_l1=0.05, min_child_samples=5
- Sampling: bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9
- **New:** 17 numerical → categorical (`_cat` suffix)

| Fold | Before (0.96000) | After num_to_cat | Change |
|------|------------------|------------------|--------|
| Fold 1 | 0.96010 | 0.96004 | -0.00006 |
| Fold 2 | 0.95897 | 0.95884 | -0.00013 |
| Fold 3 | 0.96075 | 0.96049 | -0.00026 |
| Fold 4 | 0.96022 | 0.96039 | **+0.00017** |
| Fold 5 | 0.95998 | 0.96052 | **+0.00054** |
| **OOF** | **0.96000** | **0.96004** | **+0.00004** |

**Conclusion:** Folds 1-3 slightly decreased, but folds 4-5 improved significantly. Overall OOF gained +0.00004. Feature retained for next experiment.

 🔬 Experiment: Original Dataset Target Means

Added target mean statistics from the original dataset (129,880 rows) for 21 columns (4 categorical + 17 numerical), and lowered learning rate to 0.02.

**Setup:**
- 5-fold StratifiedKFold
- Dynamic seeds: 123 + fold (124-128)
- LGBM with native categorical handling
- n_estimators=2000, early_stopping=100
- **learning_rate=0.02** (was 0.03), max_depth=8
- Regularization: lambda_l2=5.0, lambda_l1=0.05, min_child_samples=5
- Sampling: bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9
- **New:** 21 org_mean columns from original dataset

| Fold | Before (0.96004) | After org_mean | Change |
|------|------------------|----------------|--------|
| Fold 1 | 0.96004 | 0.96004 | 0.00000 |
| Fold 2 | 0.95884 | 0.95889 | **+0.00005** |
| Fold 3 | 0.96049 | 0.96054 | **+0.00005** |
| Fold 4 | 0.96039 | 0.96045 | **+0.00006** |
| Fold 5 | 0.96052 | 0.96059 | **+0.00007** |
| **OOF** | **0.96004** | **0.96010** | **+0.00006** |

**Top Features:** `Flight Distance_freq`, **`Flight Distance_org_mean`** (top-2), `TE_Flight Distance_100`, `TE_Flight Distance_auto`, `TE_Flight Distance_10`.

**Conclusion:** Original dataset target means rank high in importance (org_mean in top-2). Combined with the lower learning rate (0.02), this gave +0.00006 OOF.

 🔬 Experiment: Pair Features + TE

Added 10 pairwise interaction features and included them in the Target Encoding column set.

**Setup:**
- **New:** 10 pair features, added to TE column set

| Metric | Before | After | Gain |
|--------|--------|-------|------|
| OOF AUC | 0.96010 | **0.96031** | **+0.00021** |
| Std Fold AUC | 0.00063 | 0.00065 | — |

**Conclusion:** Pair features combined with TE on those pairs gave +0.00021 OOF. We now surpass the public XGBoost baseline (0.96018).

 🔬 Experiment: Distance Profile Features

Added distance profile features computed on `Flight Distance` (train-fold only, no leakage).

**Setup:**
- 5-fold StratifiedKFold
- Dynamic seeds: 123 + fold (124-128)
- LGBM with native categorical handling
- learning_rate=0.02, max_depth=8
- Regularization: lambda_l2=5.0, lambda_l1=0.05, min_child_samples=5
- Sampling: bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9
- **New:** Distance profile features (per `Flight Distance` route):
  - `profile_distance_count`
  - `profile_{col}_mean` and `_std` for 16 measures
  - `profile_{col}_residual` (value − route mean)
  - `profile_{cat}_{value}_share` for 4 categorical features

| Metric | Before | After | Gain |
|--------|--------|-------|------|
| OOF AUC | 0.96031 | **0.96041** | **+0.00010** |
| Std Fold AUC | 0.00065 | 0.00059 | — |

**Conclusion:** Distance profile features (route-level statistics) gave +0.00010 OOF. The features capture how a specific flight route affects satisfaction — deviations from route averages carry real signal.

🔬 Experiment: Distance Pairs in Target Encoding

Extended the TE column set with distance-based keys inspired by public XGBoost solutions: `Flight Distance × Class`, `Flight Distance × Type of Travel`, `Flight Distance × Age`, `Age × Class`, plus three distance resolutions (`div10`, `div100`, `mod1000`). Distance pairs were used **only for TE** (not as native categorical features) to avoid GPU bin-size limits.

**Setup:**
- 5-fold StratifiedKFold
- Dynamic seeds: 123 + fold (124-128)
- LGBM with native categorical handling
- learning_rate=0.02, max_depth=8
- Regularization: lambda_l2=5.0, lambda_l1=0.05, min_child_samples=5
- Sampling: bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9
- **TE columns:** 44 (21 original + 6 bins/smooth keys + 10 pairs + 4 distance pairs + 3 distance keys)
- **TE features:** 132 (44 × 3 smooth: auto, 10, 100)

| **OOF** | **0.96041** | **0.96047** | **+0.00006** |

**Conclusion:** Distance-based TE keys produced a small but consistent improvement. The `Flight Distance` axis remains the dominant signal, and multi-resolution encodings help the model capture both fine-grained and macro-level patterns along this axis.

 🧪 Experiment Log — Original-only Model Signal

**Hypothesis:**
The original 129k-row airline satisfaction dataset may contain a useful representation of the underlying satisfaction mechanism that is complementary to the synthetic Playground features.

**Change:**
Trained an auxiliary LightGBM model on the original dataset and added its predicted probability as `original_only_probability` to the Playground train/test data.

The main LightGBM pipeline remained unchanged apart from running on CPU for this experiment.

**Result:**

| Metric       | Baseline | Original-only signal |       Change |
| ------------ | -------: | -------------------: | -----------: |
| OOF AUC      |  0.96047 |          **0.96102** | **+0.00055** |
| Fold AUC Std |  0.00062 |          **0.00059** | **-0.00003** |

**Conclusion:**
The original-dataset model signal produced a meaningful OOF improvement of **+0.00055 AUC** over the baseline. The improvement is consistent across the folds rather than being caused by a single unusually strong fold.

This experiment is therefore **accepted as a promising feature family** and should be submitted to the public leaderboard for external validation.

**Next step:**
Create a submission using the current `test_preds` and compare the public leaderboard score with the baseline submission.

All experiments are conducted based on notebooks:

- [XGBoost Baseline](https://www.kaggle.com/code/ektarr/xgboost-baseline) by [Maxim](https://www.kaggle.com/ektarr)

## 📦 1. IMPORTS + SETUP

In [ ]:
import sys
sys.path.append('/kaggle/input/datasets/evgendvorkin/dvorkin-visual-v1/')
from Dvorkin_style_engine import *
set_theme("cosmic")

import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

T0 = time.perf_counter()
def log(msg):
    print(f'[{time.perf_counter() - T0:7.1f}s] {msg}', flush=True)

h1("✈️ Airline Satisfaction Prediction — S6E10")
info_card("✅ Setup Complete", "Импорты выполнены, стиль загружен.", style="vi")

## 📥 2. LOAD DATA

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/test.csv')
orig = pd.read_csv('/kaggle/input/datasets/arseniyshutko/binary-aviation-satisfaction-129k/data.csv')

h2("📥 Data Loading")
info_card("Dataset Shapes", 
          f"Train: {train.shape} | Test: {test.shape} | Orig: {orig.shape}", 
          style="vi")
pretty_df(train.head(3))

## 🔍 3. EDA — Exploratory Data Analysis


In [ ]:
def analyze_datasets(train_df, test_df, orig_df):
    datasets = {
        'train': train_df,
        'test': test_df,
        'original': orig_df
    }
    
    MAX_CAT_VALUES = 10
    
    for name, df in datasets.items():
        h2(f"{name.upper()} Dataset")
        info_card("Shape", f"Rows: {len(df)} | Columns: {len(df.columns)}", style="vi")
        
        # Missing values
        missing = df.isnull().sum()
        total_missing = missing.sum()
        
        if total_missing == 0:
            info_card("✅ Missing Values", "Нет пропусков", style="vi")
        else:
            missing_df = missing[missing > 0].reset_index()
            missing_df.columns = ['Column', 'Missing Count']
            missing_df['% Missing'] = (missing_df['Missing Count'] / len(df) * 100).round(2)
            info_card("⚠️ Missing Values", f"Всего пропусков: {total_missing} ({total_missing/df.size*100:.2f}% всех ячеек)", style="co")
            pretty_df(missing_df)
        
        # Numerical Features
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
        
        if num_cols:
            h2(f"📈 Numerical Features ({len(num_cols)})")
            desc = pd.DataFrame({
                'Column': num_cols,
                'Mean': [df[col].mean() for col in num_cols],
                'Std': [df[col].std() for col in num_cols],
                'Min': [df[col].min() for col in num_cols],
                '25%': [df[col].quantile(0.25) for col in num_cols],
                '50%': [df[col].quantile(0.5) for col in num_cols],
                '75%': [df[col].quantile(0.75) for col in num_cols],
                'Max': [df[col].max() for col in num_cols],
                'Skew': [stats.skew(df[col].dropna()) for col in num_cols]
            })
            pretty_df(desc)
            
            outlier_rows = []
            for col in num_cols:
                Q1 = df[col].quantile(0.25)
                Q3 = df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR
                outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
                outlier_rows.append({
                    'Column': col,
                    'Outliers': len(outliers),
                    'Lower Bound': round(lower_bound, 2),
                    'Upper Bound': round(upper_bound, 2)
                })
            info_card("🔍 Outliers (IQR)", "Ниже — таблица выбросов для числовых признаков", style="cy")
            pretty_df(pd.DataFrame(outlier_rows))
        
        if cat_cols:
            h2(f"📋 Categorical Features ({len(cat_cols)})")
            cat_info = pd.DataFrame({
                'Column': cat_cols,
                'Unique Values': [df[col].nunique() for col in cat_cols]
            })
            pretty_df(cat_info)
            
            for col in cat_cols:
                vc = df[col].value_counts().head(MAX_CAT_VALUES)
                if len(vc) > MAX_CAT_VALUES:
                    info_card(f"📊 {col}", f"Показаны топ {MAX_CAT_VALUES} из {df[col].nunique()} значений", style="cy")
                else:
                    info_card(f"📊 {col}", f"Всего {df[col].nunique()} уникальных значений", style="cy")
                pretty_df(pd.DataFrame({
                    'Value': vc.index,
                    'Count': vc.values,
                    '%': (vc.values / len(df) * 100).round(2)
                }))

analyze_datasets(train, test, orig)

## 🔧 4. PREPROCESSING (BASE)

In [ ]:
## 🔧 3. PREPROCESSING (BASE)

cat_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']

# Конвертируем таргет в 0/1 (bool → int)
y = train['satisfaction'].astype(int)

# Заполняем пропуски
train['Arrival Delay in Minutes'] = train['Arrival Delay in Minutes'].fillna(0)
test['Arrival Delay in Minutes'] = test['Arrival Delay in Minutes'].fillna(0)

# Категории
for col in cat_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

X = train.drop(columns=['id', 'satisfaction'])
X_test = test.drop(columns=['id'])

h2("🔧 Preprocessing")
info_card("✅ Base Preprocessing Done", 
          f"X: {X.shape} | Test: {X_test.shape} | Cat features: {len(cat_cols)}", 
          style="vi")

## 🔢 5. DIGIT FEATURES


In [ ]:
# Числовые колонки для digit (без оценок 0-5 и категорий)
digit_num_cols = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']

# Извлекаем цифры в позициях от -4 до 3
for col in digit_num_cols:
    for k in range(-4, 4):
        new_col = f'{col}_digit{k}'
        X[new_col] = (X[col].fillna(0) // (10**k) % 10).astype('int8')
        X_test[new_col] = (X_test[col].fillna(0) // (10**k) % 10).astype('int8')

print(f"✅ Добавлено digit features: {len(digit_num_cols) * 8}")
print(f"X: {X.shape} | X_test: {X_test.shape}")

## 📊 6. FREQUENCY ENCODING + RARITY


In [ ]:
# Частота + Rarity (-log freq) для всех колонок
for column in X.columns:
    freq_map = pd.concat([X[column], X_test[column]], axis=0).value_counts(normalize=True).to_dict()
    X[f'{column}_freq'] = X[column].map(freq_map).astype('float32').values
    X_test[f'{column}_freq'] = X_test[column].map(freq_map).astype('float32').values
    # Rarity = -log(freq), clip для стабильности
    X[f'{column}_rarity'] = -np.log(X[f'{column}_freq'].clip(lower=1/len(X))).astype('float32')
    X_test[f'{column}_rarity'] = -np.log(X_test[f'{column}_freq'].clip(lower=1/len(X))).astype('float32')

print(f"✅ Frequency: {len([c for c in X.columns if c.endswith('_freq')])}")
print(f"✅ Rarity: {len([c for c in X.columns if c.endswith('_rarity')])}")
print(f"X: {X.shape} | X_test: {X_test.shape}")

## 🎯 7. TRIPLE TARGET ENCODING


In [ ]:
from sklearn.preprocessing import TargetEncoder

# ============================================================
# 1. Базовые колонки для TE (21 исходный признак)
# ============================================================
te_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class',
           'Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes',
           'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
           'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
           'Inflight entertainment', 'On-board service', 'Leg room service',
           'Baggage handling', 'Checkin service', 'Cleanliness', 'distance_250_bin']

# ============================================================
# 2. Бины для TE
# ============================================================
X['age_5_bin'] = np.floor(X['Age'] / 5).astype('int32')
X_test['age_5_bin'] = np.floor(X_test['Age'] / 5).astype('int32')
X['distance_250_bin'] = np.floor(X['Flight Distance'] / 250).astype('int32')
X_test['distance_250_bin'] = np.floor(X_test['Flight Distance'] / 250).astype('int32')

# ============================================================
# 3. Smooth Keys — multi-resolution bins
# ============================================================
X['distance_100'] = np.floor(X['Flight Distance'] / 100).astype('int32')
X_test['distance_100'] = np.floor(X_test['Flight Distance'] / 100).astype('int32')
X['distance_500'] = np.floor(X['Flight Distance'] / 500).astype('int32')
X_test['distance_500'] = np.floor(X_test['Flight Distance'] / 500).astype('int32')
X['age_5'] = np.floor(X['Age'] / 5).astype('int32')
X_test['age_5'] = np.floor(X_test['Age'] / 5).astype('int32')
X['dep_delay_10'] = np.floor(X['Departure Delay in Minutes'] / 10).astype('int32')
X_test['dep_delay_10'] = np.floor(X_test['Departure Delay in Minutes'] / 10).astype('int32')

te_cols += ['age_5_bin', 'distance_250_bin', 'distance_100', 'distance_500', 'age_5', 'dep_delay_10']

# ============================================================
# 4. Пары категорий
# ============================================================
KEY_PAIRS = [
    ('Class', 'Type of Travel'), ('Customer Type', 'Type of Travel'), ('Class', 'Customer Type'),
    ('Gender', 'Class'), ('Inflight wifi service', 'Online boarding'),
    ('Inflight wifi service', 'Type of Travel'), ('Online boarding', 'Type of Travel'),
    ('Seat comfort', 'Inflight entertainment'), ('Class', 'Inflight wifi service'),
    ('Class', 'Online boarding'),
]

pair_cols = []
for c1, c2 in KEY_PAIRS:
    pair_name = f'pair_{c1}_{c2}'
    X[pair_name] = (X[c1].astype(str) + '_' + X[c2].astype(str)).astype('category')
    X_test[pair_name] = (X_test[c1].astype(str) + '_' + X_test[c2].astype(str)).astype('category')
    pair_cols.append(pair_name)

te_cols += pair_cols

# ============================================================
# 5. Distance Pairs + Age×Class — ТОЛЬКО для TE (не в cat_cols!)
# ============================================================
DISTANCE_PAIRS_OTHERS = ['Class', 'Type of Travel', 'Age']
distance_pair_cols = []
for other in DISTANCE_PAIRS_OTHERS:
    name = f'new_distance__{other}'
    X[name] = (X['Flight Distance'].astype(str) + '_' + X[other].astype(str)).astype('category')
    X_test[name] = (X_test['Flight Distance'].astype(str) + '_' + X_test[other].astype(str)).astype('category')
    distance_pair_cols.append(name)

X['new_age__class'] = (X['Age'].astype(str) + '_' + X['Class'].astype(str)).astype('category')
X_test['new_age__class'] = (X_test['Age'].astype(str) + '_' + X_test['Class'].astype(str)).astype('category')
distance_pair_cols.append('new_age__class')

# ============================================================
# 6. Distance Keys: div10, div100, mod1000
# ============================================================
X['new_distance_div10'] = np.floor(X['Flight Distance'] / 10).astype('int32')
X_test['new_distance_div10'] = np.floor(X_test['Flight Distance'] / 10).astype('int32')
X['new_distance_div100'] = np.floor(X['Flight Distance'] / 100).astype('int32')
X_test['new_distance_div100'] = np.floor(X_test['Flight Distance'] / 100).astype('int32')
X['new_distance_mod1000'] = (X['Flight Distance'] % 1000).astype('int32')
X_test['new_distance_mod1000'] = (X_test['Flight Distance'] % 1000).astype('int32')

distance_key_cols = ['new_distance_div10', 'new_distance_div100', 'new_distance_mod1000']

te_cols += distance_pair_cols + distance_key_cols

# ============================================================
# 7. cat_cols — ТОЛЬКО исходные категории + пары
#    Distance pairs НЕ добавляем (иначе GPU падает из-за 19471 категории)
# ============================================================
cat_cols = cat_cols + pair_cols

# ============================================================
# 8. Читаем raw train/test для TE
# ============================================================
train_te_raw = pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/train.csv')
test_te_raw = pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/test.csv')

# Бины и smooth keys в raw
for col_name in ['age_5_bin', 'distance_250_bin', 'distance_100', 'distance_500', 'age_5', 'dep_delay_10']:
    train_te_raw[col_name] = X[col_name].values
    test_te_raw[col_name] = X_test[col_name].values

# Пары в raw
for pair_name, (c1, c2) in zip(pair_cols, KEY_PAIRS):
    train_te_raw[pair_name] = train_te_raw[c1].astype(str) + '_' + train_te_raw[c2].astype(str)
    test_te_raw[pair_name] = test_te_raw[c1].astype(str) + '_' + test_te_raw[c2].astype(str)

# Distance pairs в raw
for other in DISTANCE_PAIRS_OTHERS:
    name = f'new_distance__{other}'
    train_te_raw[name] = train_te_raw['Flight Distance'].astype(str) + '_' + train_te_raw[other].astype(str)
    test_te_raw[name] = test_te_raw['Flight Distance'].astype(str) + '_' + test_te_raw[other].astype(str)

train_te_raw['new_age__class'] = train_te_raw['Age'].astype(str) + '_' + train_te_raw['Class'].astype(str)
test_te_raw['new_age__class'] = test_te_raw['Age'].astype(str) + '_' + test_te_raw['Class'].astype(str)

# Distance keys в raw
train_te_raw['new_distance_div10'] = np.floor(train_te_raw['Flight Distance'] / 10).astype('int32')
test_te_raw['new_distance_div10'] = np.floor(test_te_raw['Flight Distance'] / 10).astype('int32')
train_te_raw['new_distance_div100'] = np.floor(train_te_raw['Flight Distance'] / 100).astype('int32')
test_te_raw['new_distance_div100'] = np.floor(test_te_raw['Flight Distance'] / 100).astype('int32')
train_te_raw['new_distance_mod1000'] = (train_te_raw['Flight Distance'] % 1000).astype('int32')
test_te_raw['new_distance_mod1000'] = (test_te_raw['Flight Distance'] % 1000).astype('int32')

print(f"✅ TE колонок: {len(te_cols)} × 3 smooth = {len(te_cols) * 3} TE-признаков")
print(f"✅ Distance pairs: {len(distance_pair_cols)}")
print(f"✅ Distance keys: {len(distance_key_cols)}")
print(f"✅ cat_cols: {len(cat_cols)}")
print(f"X: {X.shape} | X_test: {X_test.shape}")

## 🔑 8. SMOOTH KEYS


In [ ]:

# Smooth Keys — multi-resolution bins для числовых
X['distance_100'] = np.floor(X['Flight Distance'] / 100).astype('int32')
X_test['distance_100'] = np.floor(X_test['Flight Distance'] / 100).astype('int32')

X['distance_500'] = np.floor(X['Flight Distance'] / 500).astype('int32')
X_test['distance_500'] = np.floor(X_test['Flight Distance'] / 500).astype('int32')

X['age_5'] = np.floor(X['Age'] / 5).astype('int32')
X_test['age_5'] = np.floor(X_test['Age'] / 5).astype('int32')

X['dep_delay_10'] = np.floor(X['Departure Delay in Minutes'] / 10).astype('int32')
X_test['dep_delay_10'] = np.floor(X_test['Departure Delay in Minutes'] / 10).astype('int32')

print(f"✅ Smooth Keys добавлены: 4")
print(f"X: {X.shape} | X_test: {X_test.shape}")

## 🔢 9. NUMERIC TO CATEGORICAL

In [ ]:
# Числовые колонки → категории
num_to_cat_cols = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes'] + \
                  ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
                   'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
                   'Inflight entertainment', 'On-board service', 'Leg room service',
                   'Baggage handling', 'Checkin service', 'Cleanliness']

for col in num_to_cat_cols:
    cat_name = f'{col}_cat'
    X[cat_name] = X[col].astype('int32').astype('category')
    X_test[cat_name] = X_test[col].astype('int32').astype('category')

print(f"✅ Num-to-cat добавлено: {len(num_to_cat_cols)}")
print(f"X: {X.shape} | X_test: {X_test.shape}")

## 🔗 10. PAIR FEATURES

In [ ]:
KEY_PAIRS = [
    ('Class', 'Type of Travel'), ('Customer Type', 'Type of Travel'), ('Class', 'Customer Type'),
    ('Gender', 'Class'), ('Inflight wifi service', 'Online boarding'),
    ('Inflight wifi service', 'Type of Travel'), ('Online boarding', 'Type of Travel'),
    ('Seat comfort', 'Inflight entertainment'), ('Class', 'Inflight wifi service'),
    ('Class', 'Online boarding'),
]

pair_cols = []
for c1, c2 in KEY_PAIRS:
    pair_name = f'pair_{c1}_{c2}'
    X[pair_name] = (X[c1].astype(str) + '_' + X[c2].astype(str)).astype('category')
    X_test[pair_name] = (X_test[c1].astype(str) + '_' + X_test[c2].astype(str)).astype('category')
    pair_cols.append(pair_name)

print(f"✅ Pair features добавлены: {len(pair_cols)}")
print(f"X: {X.shape} | X_test: {X_test.shape}")

## 📊 11. RARITY FOR PAIRS

In [ ]:
for col in pair_cols:
    freq_map = pd.concat([X[col], X_test[col]], axis=0).value_counts(normalize=True).to_dict()
    freq_values = X[col].map(freq_map).astype('float32').values
    X[f'{col}_rarity'] = -np.log(np.clip(freq_values, 1/len(X), 1)).astype('float32')
    freq_values_test = X_test[col].map(freq_map).astype('float32').values
    X_test[f'{col}_rarity'] = -np.log(np.clip(freq_values_test, 1/len(X), 1)).astype('float32')

print(f"✅ Rarity для пар: {len(pair_cols)}")
print(f"X: {X.shape} | X_test: {X_test.shape}")

## 🎯 12. ORIGINAL DATASET TARGET MEANS


In [ ]:
# Оригинальный датасет — target encoding
orig['satisfaction'] = orig['satisfaction'].map({False: 0, True: 1})
orig_global_mean = orig['satisfaction'].mean()

# Колонки для org_mean: 4 категории + 17 числовых
org_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class',
            'Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes',
            'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
            'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
            'Inflight entertainment', 'On-board service', 'Leg room service',
            'Baggage handling', 'Checkin service', 'Cleanliness']

# Читаем raw train/test
train_org_raw = pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/train.csv')
test_org_raw = pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/test.csv')

# Считаем org_mean по каждой колонке
for col in org_cols:
    if col in orig.columns:
        stats = orig.groupby(col, observed=False)['satisfaction'].mean()
        X[f'{col}_org_mean'] = train_org_raw[col].map(stats).fillna(orig_global_mean).astype('float32').values
        X_test[f'{col}_org_mean'] = test_org_raw[col].map(stats).fillna(orig_global_mean).astype('float32').values

print(f"✅ Org mean добавлено: {len(org_cols)}")
print(f"X: {X.shape} | X_test: {X_test.shape}")

del train_org_raw, test_org_raw
gc.collect()

## 13. ORIGINAL-ONLY MODEL SIGNAL

Train a separate LightGBM model on the original 129k-row dataset and transfer its predicted probability to the Playground train/test rows.

The hypothesis is that the original dataset contains a useful representation of the underlying satisfaction mechanism that can complement the synthetic Playground features.


In [ ]:
# Original-only model
# This model is trained exclusively on the external original dataset.
# It does not use labels from the Playground competition data.

from lightgbm import LGBMClassifier
import numpy as np

ORIG_MODEL_FEATURES = [
    'Gender',
    'Customer Type',
    'Age',
    'Type of Travel',
    'Class',
    'Flight Distance',
    'Inflight wifi service',
    'Departure/Arrival time convenient',
    'Ease of Online booking',
    'Gate location',
    'Food and drink',
    'Online boarding',
    'Seat comfort',
    'Inflight entertainment',
    'On-board service',
    'Leg room service',
    'Baggage handling',
    'Checkin service',
    'Cleanliness',
    'Departure Delay in Minutes',
    'Arrival Delay in Minutes',
]

orig_model = orig[ORIG_MODEL_FEATURES].copy()
orig_target = orig['satisfaction'].astype('int8')

# Convert categorical columns to category dtype.
ORIG_CAT_COLS = [
    'Gender',
    'Customer Type',
    'Type of Travel',
    'Class',
]

for col in ORIG_CAT_COLS:
    orig_model[col] = orig_model[col].astype('category')

# Train an auxiliary LightGBM model on the original dataset only.
orig_model_params = {
    'objective': 'binary',
    'n_estimators': 600,
    'learning_rate': 0.05,
    'max_depth': 8,          # maximum tree depth
    'min_child_samples': 20, # minimum samples in a leaf
    'subsample': 0.8,         # row sampling ratio
    'colsample_bytree': 0.5, # feature sampling ratio
    'reg_lambda': 5.0,        # L2 regularization
    'reg_alpha': 0.05,        # L1 regularization
    'random_state': 123,
    'n_jobs': -1,
    'verbosity': -1,
}

orig_model_lgb = LGBMClassifier(**orig_model_params)

orig_model_lgb.fit(
    orig_model,
    orig_target,
    categorical_feature=ORIG_CAT_COLS,
)

# Prepare competition train/test features.
orig_train_model = train[ORIG_MODEL_FEATURES].copy()
orig_test_model = test[ORIG_MODEL_FEATURES].copy()

for col in ORIG_CAT_COLS:
    orig_train_model[col] = orig_train_model[col].astype('category')
    orig_test_model[col] = orig_test_model[col].astype('category')

# Align categorical levels with the original-data model.
for col in ORIG_CAT_COLS:
    categories = orig_model[col].cat.categories

    orig_train_model[col] = (
        orig_train_model[col]
        .astype(pd.CategoricalDtype(categories=categories))
    )
    orig_test_model[col] = (
        orig_test_model[col]
        .astype(pd.CategoricalDtype(categories=categories))
    )

# Model probability.
orig_train_prob = orig_model_lgb.predict_proba(orig_train_model)[:, 1]
orig_test_prob = orig_model_lgb.predict_proba(orig_test_model)[:, 1]

# Convert probability to logit.
eps = 1e-6

orig_train_logit = np.log(
    np.clip(orig_train_prob, eps, 1 - eps)
    / np.clip(1 - orig_train_prob, eps, 1 - eps)
)

orig_test_logit = np.log(
    np.clip(orig_test_prob, eps, 1 - eps)
    / np.clip(1 - orig_test_prob, eps, 1 - eps)
)

X['original_only_probability'] = orig_train_prob
X['original_only_logit'] = orig_train_logit

X_test['original_only_probability'] = orig_test_prob
X_test['original_only_logit'] = orig_test_logit

print(
    'Added original-only model features:',
    ['original_only_probability', 'original_only_logit']
)

print('X shape:', X.shape)
print('X_test shape:', X_test.shape)

## 📏 14. DISTANCE PROFILE

In [ ]:
# Flight Distance — ключ маршрута
distance = 'Flight Distance'

# Колонки для профиля
profile_measures = ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
                    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
                    'Inflight entertainment', 'On-board service', 'Leg room service',
                    'Baggage handling', 'Checkin service', 'Cleanliness',
                    'Age', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']

profile_cat_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']

print(f"✅ Distance Profile setup готов")
print(f"Measures: {len(profile_measures)} | Categorical: {len(profile_cat_cols)}")

## 🚀 15. LIGHTGBM

In [ ]:
import lightgbm as lgb

params = {
    'objective': 'binary',              # Binary classification
    'metric': 'auc',                    # ROC AUC metric
    'n_estimators': 2000,               # Max boosting rounds
    'learning_rate': 0.02,              # Step size shrinkage
    'max_depth': 8,                     # Max tree depth
    'random_state': 123,                # Random seed
    'n_jobs': -1,                       # All CPU threads
    'verbose': -1,                      # Suppress LGBM INFO/warnings
    'early_stopping_rounds': 100,       # Stop if no improvement
    'lambda_l2': 5.0,                   # L2 regularization on leaf weights
    'lambda_l1': 0.05,                  # L1 regularization for feature selection
    'min_child_samples': 5,             # Minimum samples per leaf
    'bagging_fraction': 0.9,            # Row sampling per iteration
    'bagging_freq': 1,                  # Apply bagging every iteration
    'feature_fraction': 0.9,            # Feature sampling per tree
    'device': 'cpu',                    # GPU acceleration
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
feature_importance = None
fold_scores = []

log("Starting 5-fold CV with Triple TE + Distance Profile")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train = X.iloc[train_idx].copy()
    X_val = X.iloc[val_idx].copy()
    X_test_fold = X_test.copy()
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]
    
    # Triple Target Encoding (auto, 10, 100) по te_cols
    for smooth_val, smooth_name in [('auto', 'auto'), (10.0, '10'), (100.0, '100')]:
        te = TargetEncoder(shuffle=True, cv=5, smooth=smooth_val, random_state=42)
        X_train_enc = te.fit_transform(train_te_raw.iloc[train_idx][te_cols], y_train).astype('float32')
        X_val_enc = te.transform(train_te_raw.iloc[val_idx][te_cols]).astype('float32')
        X_test_enc = te.transform(test_te_raw[te_cols]).astype('float32')
        
        for i, col in enumerate(te_cols):
            X_train[f'TE_{col}_{smooth_name}'] = X_train_enc[:, i]
            X_val[f'TE_{col}_{smooth_name}'] = X_val_enc[:, i]
            X_test_fold[f'TE_{col}_{smooth_name}'] = X_test_enc[:, i]
    
    # Distance Profile — fitted on training rows only
    train_fold_dist = X.iloc[train_idx][distance]
    grouped = X.iloc[train_idx].groupby(distance, observed=True)[profile_measures].agg(['mean', 'std'])
    counts = train_fold_dist.value_counts()
    defaults = {col: {'mean': float(X.iloc[train_idx][col].mean()),
                      'std': float(X.iloc[train_idx][col].std())} for col in profile_measures}
    
    proportions = {}
    for col in profile_cat_cols:
        proportions[col] = pd.crosstab(X.iloc[train_idx][distance], X.iloc[train_idx][col], normalize='index')
    
    for frame in [X_train, X_val, X_test_fold]:
        frame['profile_distance_count'] = frame[distance].map(counts).fillna(0).astype('float32')
        for col in profile_measures:
            name = col.lower().replace(' ', '_').replace('/', '_').replace('-', '_')
            frame[f'profile_{name}_mean'] = frame[distance].map(grouped[(col, 'mean')]).fillna(defaults[col]['mean']).astype('float32')
            frame[f'profile_{name}_std'] = frame[distance].map(grouped[(col, 'std')]).fillna(defaults[col]['std']).astype('float32')
            frame[f'profile_{name}_residual'] = (frame[col] - frame[f'profile_{name}_mean']).astype('float32')
        for col in profile_cat_cols:
            prior = X.iloc[train_idx][col].value_counts(normalize=True)
            name_col = col.lower().replace(' ', '_')
            for cat_val in proportions[col].columns:
                name = f'profile_{name_col}_{str(cat_val).lower().replace(" ", "_")}_share'
                frame[name] = frame[distance].map(proportions[col][cat_val]).fillna(prior[cat_val]).astype('float32')
    
    current_seed = 123 + fold
    params['random_state'] = current_seed
    
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        categorical_feature=cat_cols,
        callbacks=[lgb.log_evaluation(period=100)]
    )
    
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test_fold)[:, 1] / skf.n_splits
    
    if feature_importance is None:
        feature_importance = np.zeros(X_train.shape[1])
    feature_importance += model.feature_importances_ / skf.n_splits
    
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    fold_scores.append(fold_auc)
    log(f"Fold {fold}/5: AUC = {fold_auc:.5f} | Best iter: {model.best_iteration_} (seed {current_seed})")

oof_auc = roc_auc_score(y, oof_preds)

h2("🏆 Final OOF Performance")
fold_df = pd.DataFrame({
    'Fold': [f'Fold {i}' for i in range(1, 6)],
    'AUC': [f'{s:.5f}' for s in fold_scores],
    'Seed': [123 + i for i in range(1, 6)]
})
pretty_df(fold_df)
md_metric("OOF AUC", f"{oof_auc:.5f}")
md_metric("Std Fold AUC", f"{np.std(fold_scores):.5f}")

# Сохраняем feature importance для финального анализа
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': feature_importance
}).sort_values('importance', ascending=False).reset_index(drop=True)

## 📤 16. CREATE SUBMISSION


In [ ]:
submission = pd.DataFrame({
    'id': pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/test.csv')['id'],
    'satisfaction': test_preds
})

submission.to_csv('submission.csv', index=False)

h2("📤 Submission Created")
info_card("✅ Submission Saved", f"Shape: {submission.shape}", style="vi")
pretty_df(submission.head(10))

## 📊17. Complete Feature Importance

In [ ]:
h2(f"📊 Complete Feature Importance — All {len(importance_df)} Features")
info_card("Analysis", f"Averaged over 5 folds | Sorted by importance (highest → lowest)", style="vi")

display(importance_df.style
    .format({'importance': '{:.4f}', 'cumulative': '{:.1%}'})
    .bar(subset=['importance'], color='#2563eb')
    .set_caption(f'Feature Importance — {len(importance_df)} features')
)